# Simulation

In [1]:
# Data handling: polars is used instead of pandas because the sales/price tables run to
# millions of rows and the reshape + join steps below are much cheaper on its query engine.
import polars as pl
from pathlib import Path

# Modelling: sklearn for scaling/PCA, scipy for the kernel density estimate of WTP.
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import numpy as np
import scipy
import heapq
from collections import defaultdict
# Plotting.
import seaborn as sns
import matplotlib.pyplot as plt

from sim import Demand, Env, Inventory, SupplierInfo, Supplier

## Import data

The three raw M5 CSVs are read straight from `data/` at the repo root. Their schemas are documented
below so the joins further down (`d` for daily grain, `wm_yr_wk` for weekly price grain) are easy to follow.

## M5 Forecasting Dataset Column Definitions

------------------------------
## 📅 calendar.csv
This file links daily sales data to specific dates, financial weeks, calendar events, and social assistance schedules.

* date: The calendar date formatted as YYYY-MM-DD.
* wm_yr_wk: A unique ID indicating the Walmart financial week. It is used to join this file with sell_prices.csv.
* weekday: The name of the day (e.g., Saturday, Sunday).
* wday: A numerical ID representing the day of the week, starting with Saturday (1) through Friday (7).
* month: The numerical month of the year (1–12).
* year: The numerical year.
* d: The specific day identifier used as headers in the training sets (e.g., d_1, d_2 ... up to d_1969).
* event_name_1: The name of a special holiday or cultural event happening on that day (e.g., SuperBowl, Thanksgiving, EidAlFitr). It is blank if no event occurs.
* event_type_1: The category classification of event_name_1 (e.g., Sporting, Cultural, National, Religious).
* event_name_2: Used on rare days where two distinct public events overlap on the same date.
* event_type_2: The category classification of event_name_2.
* snap_CA, snap_TX, snap_WI: Binary flags (0 or 1) indicating whether the stores in California, Texas, or Wisconsin allow SNAP (food stamp) purchases on that specific day.

------------------------------
## 📈 sales_train_validation.csv / sales_train_evaluation.csv
These horizontal datasets contain the historical unit transaction columns that establish the product hierarchy.

* id: A concatenated unique text key representing the specific product identifier combined with the store ID (e.g., HOBBIES_1_001_CA_1_validation).
* item_id: The granular code assigned to the product item itself (e.g., HOBBIES_1_001).
* dept_id: The department code separating broad categories (e.g., HOBBIES_1, FOODS_3).
* cat_id: The top-level category label (Hobbies, Foods, or Household).
* store_id: The store identifier where the product was sold (e.g., CA_1, TX_3, WI_2).
* state_id: The broader geographical US region (CA, TX, or WI).
* d_1 to d_1941: Sequential column headers tracking the exact quantity of units sold on that corresponding day index. This maps directly to the d column in calendar.csv.

------------------------------
## 🏷️ sell_prices.csv
This table allows you to extract revenue and price elasticity metrics across different stores.

* store_id: The store location code matching the sales matrix.
* item_id: The specific product identifier matching the sales matrix.
* wm_yr_wk: The target Walmart financial week index matching calendar.csv.
* sell_price: The specific retail dollar price of the given item at that store location during that operational week. Prices are tracked weekly because retail prices typically stay fixed for 7-day spans.






In [2]:
data_folder = Path.cwd().parents[1] / 'data'

# --- ingestion: no data is read here; scan_csv only inspects the header ---
calendar_lf = pl.scan_csv(data_folder / 'calendar.csv')
prices_lf   = pl.scan_csv(data_folder / 'sell_prices.csv')
sales_lf    = pl.scan_csv(data_folder / 'sales_train_evaluation.csv')

In [3]:
STORE = ['CA_1', 'CA_2']

# collect_schema() reads the header only, it does not execute the plan.
day_cols  = [c for c in sales_lf.collect_schema().names() if c.startswith('d_')]
day_order = sorted(day_cols, key=lambda c: int(c.removeprefix('d_')))   # chronological, not lexical

# --- demand: wide daily matrix -> long (item_id, d, demand) + financial week ---
demand_lf = (
    sales_lf
    .filter(pl.col('store_id').is_in(STORE))          # predicate pushed into the scan
    .select(['item_id', *day_cols])
    .unpivot(index='item_id', on=day_cols, variable_name='d', value_name='demand')
    .join(calendar_lf.select('d', 'wm_yr_wk'), on='d', how='inner')
)

# --- prices: same store, join keys + price only (projection pushdown drops the rest) ---
prices_store_lf = (
    prices_lf
    .filter(pl.col('store_id').is_in(STORE))
    .select('item_id', 'wm_yr_wk', 'sell_price')
) 

# --- item-day fact table ---
merged_lf = (
    demand_lf
    .join(prices_store_lf, on=['item_id', 'wm_yr_wk'], how='inner')   # inner: drops unpriced item-weeks
    .with_columns(
        pl.col('item_id').str.split('_').list.slice(0, 2).list.join('_').alias('dept_id'),
        pl.col('d').str.split('_').list.get(-1).cast(pl.Int32).alias('day_n'),
    )
)

# --- department x day aggregate ---
dept_day_lf = (
    merged_lf
    .group_by('dept_id', 'day_n')
    .agg(
        pl.col('demand').sum().alias('total_demand'),
        (pl.col('demand') * pl.col('sell_price')).sum().alias('total_weighted_price'),
    )
    .with_columns(
        (pl.col('total_weighted_price') / pl.col('total_demand')).round(3).alias('price_avg')
    )
    .filter(pl.col('price_avg').is_not_nan())     # zero-demand department-days give 0/0
)

wide_df = (
    dept_day_lf
    .sort('day_n')                                # column order = order of first appearance
    .collect()
    .pivot(index='dept_id', on='day_n', values='total_demand')
)

# Pivot on the integer day, then relabel — the columns are already chronological,
# so this rename maps each column onto its own data.
wide_df.columns = ['dept_id'] + [f'd_{c}' for c in wide_df.columns[1:]]

day_spine = pl.LazyFrame(
    {'day_n': [int(c.removeprefix('d_')) for c in day_order]},
    schema={'day_n': pl.Int32},
)

wide_lf = (
    dept_day_lf.select('dept_id').unique()
    .join(day_spine, how='cross')                                     # every dept x every day
    .join(dept_day_lf.select('dept_id', 'day_n', 'total_demand'),
          on=['dept_id', 'day_n'], how='left')
    .with_columns(pl.col('total_demand').fill_null(0))
    .group_by('dept_id')
    .agg(pl.col('total_demand').sort_by('day_n'))                     # explicit ordering
    .with_columns(pl.col('total_demand').list.to_struct(fields=day_order))
    .unnest('total_demand')
    .sort(by='dept_id')
)

wide_df = wide_lf.collect()          # .collect(engine="streaming") if memory is tight


## Demand

### Demand  willingness to pay

In [4]:
# --- demand side: wide daily matrix -> long (item_id, d, demand) ---
demand_ca_1 = sales_lf.lazy().filter(pl.col('store_id').is_in(STORE))

# Every column named d_* is a day of history; the rest are hierarchy/metadata columns.
# collect_schema() reads the header only, doesn't execute the plan.
day_cols = [col for col in sales_lf.collect_schema().names() if col.startswith('d_')]

demand_ca_1 = demand_ca_1.select(['item_id'] + day_cols).unpivot(
    index='item_id',
    on=day_cols,
    variable_name='d',
    value_name='demand'
)

# Attach the Walmart financial week for each day so the weekly prices can be joined on.
demand_ca_1 = demand_ca_1.join(
    calendar_lf.lazy().select(pl.col('d', 'wm_yr_wk')),
    on='d',
    how='inner'
)

# --- price side: same store, keep only the join keys and the price itself ---
prices_ca_1 = prices_lf.lazy().filter(
    pl.col('store_id').is_in(STORE)
).select(
    pl.col('item_id', 'wm_yr_wk', 'sell_price')
)

# Inner join drops item-days with no active price (item not stocked that week) — see note above.
merge = demand_ca_1.join(prices_ca_1, on=['item_id', 'wm_yr_wk'], how='inner')

# Recover the department from the item code: HOBBIES_1_001 -> HOBBIES_1 (first two underscore tokens).
merge = merge.with_columns(
    pl.col("item_id").str.split("_").list.slice(0, 2).list.join("_").alias("dept_id")
)

# Roll the item-level table up to one row per (day, department).
ca_1_lf = merge.with_columns(
    (pl.col('demand') * pl.col('sell_price')).alias('total_price')
).group_by(
    ['d', 'dept_id']
).agg(
    pl.col('demand').sum().alias('total_demand'),
    pl.col('total_price').sum().alias('total_weighted_price')
).with_columns(
    (pl.col('total_weighted_price') / pl.col('total_demand')).round(3).alias('price_avg'),
    pl.col('d').str.split("_").list.get(-1).cast(pl.Int32).alias('day_n')
).drop_nans(subset='price_avg')

# Nothing executes until collect() is called — this triggers predicate/projection pushdown
# (filters and column selection happen as early as possible in the scan).
ca_1 = ca_1_lf.collect()
print(ca_1.shape)
ca_1

(13557, 6)


d,dept_id,total_demand,total_weighted_price,price_avg,day_n
str,str,i64,f64,f64,i32
"""d_1883""","""FOODS_2""",1938,7394.7,3.816,1883
"""d_1810""","""FOODS_2""",1562,5754.86,3.684,1810
"""d_1024""","""HOUSEHOLD_2""",1062,5561.82,5.237,1024
"""d_1419""","""HOUSEHOLD_1""",1631,6811.89,4.177,1419
"""d_1934""","""HOUSEHOLD_1""",4670,20170.87,4.319,1934
…,…,…,…,…,…
"""d_1693""","""HOBBIES_2""",170,306.5,1.803,1693
"""d_1693""","""FOODS_1""",1240,3498.55,2.821,1693
"""d_1695""","""HOUSEHOLD_2""",1326,6372.3,4.806,1695


In [5]:
wtp_kde = {
    df['dept_id'][0]: scipy.stats.gaussian_kde(df['price_avg'], weights=df['total_demand'])
    for df in ca_1.partition_by('dept_id')
}

# 1-wtp_kde['HOUSEHOLD_1'].integrate_box_1d(-np.inf, 3.5)

### Demand module

In [23]:
demand = Demand(wide_df, wtp_kde)

In [24]:
demand.generate_demand()

array([[ 1445.,  1301.,   973., ...,  1769.,  2285.,  2225.],
       [ 1452.,  1612.,   918., ...,  1901.,  2982.,  3260.],
       [ 6223.,  5961.,  3732., ...,  8909., 10982., 11678.],
       ...,
       [   73.,    51.,    29., ...,   171.,   280.,   290.],
       [ 1832.,  1638.,  1437., ...,  3062.,  4748.,  4961.],
       [  885.,   906.,   589., ...,  1026.,  1795.,  1556.]],
      shape=(7, 1941))

In [33]:
demand.product

['FOODS_1',
 'FOODS_2',
 'FOODS_3',
 'HOBBIES_1',
 'HOBBIES_2',
 'HOUSEHOLD_1',
 'HOUSEHOLD_2']

In [25]:
demand_now = iter(demand.generate_demand().T.astype(int))

In [32]:
next(demand_now)

array([ 805,  784, 4209, 1047,   38, 1089,  569])

In [9]:
np.hstack([np.array(demand.product).reshape(-1,1), next(demand_now).reshape(-1,1)])

array([['FOODS_1', '1445'],
       ['FOODS_2', '1452'],
       ['FOODS_3', '6223'],
       ['HOBBIES_1', '1942'],
       ['HOBBIES_2', '73'],
       ['HOUSEHOLD_1', '1832'],
       ['HOUSEHOLD_2', '885']], dtype='<U21')

## Enviroment

In [10]:
sim = Env()

## Inventory

In [11]:
initial_order = {
    'FOODS_1': {
        'quantity': 5000,
        'price': 3,   
        'shelf_life': 60,
    },
    'FOODS_2': {
        'quantity': 5000,
        'price': 3,   
        'shelf_life': 60,
    },
    'FOODS_3': {
        'quantity': 5000,
        'price': 3,   
        'shelf_life': 60,
    },
    'HOBBIES_1': {
        'quantity': 5000,
        'price': 3,   
        'shelf_life': 60,
    },
    'HOBBIES_2': {
        'quantity': 5000,
        'price': 3,   
        'shelf_life': 60,
    },
    'HOUSEHOLD_1': {
        'quantity': 5000,
        'price': 3,   
        'shelf_life': 60,
    },
    'HOUSEHOLD_2': {
        'quantity': 5000,
        'price': 3,   
        'shelf_life': 60,
    },
}

In [12]:
inventory = Inventory(sim, demand.wtp)

In [13]:
inventory.stockup(initial_order)
inventory.set_price(initial_order)

In [14]:
inventory.stock['FOODS_2'].sku

[[60, 5000]]

In [15]:
inventory.inventory_status()

[['FOODS_1',
  'FOODS_2',
  'FOODS_3',
  'HOBBIES_1',
  'HOBBIES_2',
  'HOUSEHOLD_1',
  'HOUSEHOLD_2'],
 [5000, 5000, 5000, 5000, 5000, 5000, 5000],
 [3, 3, 3, 3, 3, 3, 3]]

In [16]:
demand_order = {
    'FOODS_1': 500,
    'HOBBIES_2': 3000,
    'HOUSEHOLD_2': 6000
}

In [17]:
inventory.sell_stock(demand_order)

[['FOODS_1', 'HOBBIES_2', 'HOUSEHOLD_2'], [6, 93, 5000]]

In [18]:
inventory.inventory_status()

[['FOODS_1',
  'FOODS_2',
  'FOODS_3',
  'HOBBIES_1',
  'HOBBIES_2',
  'HOUSEHOLD_1',
  'HOUSEHOLD_2'],
 [4994, 5000, 5000, 5000, 4907, 5000, 0],
 [3, 3, 3, 3, 3, 3, 3]]

## Supplier

In [19]:
supplier_book = Supplier()

# Freeze the distributions with explicit loc= keywords: order_proposal reads
# lead_time.kwds['loc'], and freezing positionally leaves .kwds empty -> KeyError.
#
# price is a volume price break — one distribution per entry in order_quantity, so the
# unit cost falls as the lot size rises. The two lists must be the same length.
# These are wholesale costs; FOODS_2 retails at roughly $3.50-3.70 in the data.
supplier_book.add_supplier(
    SupplierInfo(
        name='bulk_grocer',
        product='FOODS_2',
        price=[scipy.stats.norm(loc=2.30, scale=0.08),
               scipy.stats.norm(loc=2.10, scale=0.08),
               scipy.stats.norm(loc=1.95, scale=0.08)],
        lead_time=scipy.stats.norm(loc=7, scale=1.5),
        order_quantity=[5000, 10000, 20000],
    ),
)

# order_proposal takes a bare code or a collection of them, and returns a list of Quote.
quotes = supplier_book.order_proposal('FOODS_2')

for q in quotes:
    print(f'{q.supplier.name}  ({q.product}, lead {q.lead_time} days)')
    for lot in q.order_quantity:
        print(f'    {lot:>6,} units @ {q.price_for(lot):.2f} = {q.cost_for(lot):>10,.2f}')

bulk_grocer  (FOODS_2, lead 7 days)
     5,000 units @ 2.43 =  12,150.00
    10,000 units @ 2.18 =  21,800.00
    20,000 units @ 1.88 =  37,600.00


In [38]:
# With volume price breaks the decision is (supplier, lot size), not just supplier — so
# search over every offered lot. Cheapest unit price is a placeholder rule: it always
# picks the largest lot, ignoring the holding cost and spoilage that come with it. That
# tradeoff is what the Policy section has to solve.
best, best_lot = min(
    ((q, lot) for q in quotes for lot in q.order_quantity),
    key=lambda pair: pair[0].price_for(pair[1]),
)

# order_purchase wants the accepted subset keyed by the SupplierInfo object, with the
# unit price for the lot size chosen. It rejects a quantity the supplier does not offer.
supplier_book.order_purchase(
    sim,
    {best.supplier: {'quantity': best_lot, 'price': best.price_for(best_lot)}},
    demand_index = 0.8
)

print(f'accepted {best.supplier.name}: {best_lot:,} units @ '
      f'{best.price_for(best_lot):.2f} = {best.cost_for(best_lot):,.2f}')
print('clock:', sim.clock)
print('calendar:', sim.event_calendar)   # (arrival_day, 'stockup', product, quantity, price)

accepted bulk_grocer: 20,000 units @ 1.88 = 37,600.00
clock: 0
calendar: [(14, 'stockup', 'FOODS_2', 20000, np.float64(1.88))]


## Policy

## Simulation

In [ ]:
EPOCH = 100
RUN_DAYS = 365

PRODUCT = demand.product

shelf_life_dict = {
    'FOODS_1'       : 70,
    'FOODS_2'       : 35,
    'FOODS_3'       : 15,
    'HOBBIES_1'     : 85,
    'HOBBIES_2'     : 100,
    'HOUSEHOLD_1'   : 400,
    'HOUSEHOLD_2'   : 400,
}
wastage = []

In [ ]:
# Process event
def process_events(time, event):

    if event[0] == 'stockup':
        inventory.stockup({event[1]: {'quantity': event[2], 'shelf_life': shelf_life_dict[event[1]]}})
        return None

    if event[0] == 'expire':
        flushout = inventory.flushout_expired(event[1])
        wastage.append((time, flushout))

In [ ]:
for epoch in range(EPOCH):
    sim.reset()
    demand_data = demand.generate_demand().T.astype(int)
    demand_gen = iter(demand_data)
    while sim.clock <= RUN_DAYS:
        todays_demand = next(demand_gen)
        if sim.clock in sim.event_calendar:
            for event in sim.event_calendar[sim.clock]:
                process_events(sim.clock, event) # process event
            sim.remove_event(sim.clock)
            
                    
        # policy place holder
        
        sim.clock_step()